# Step 4B: End-to-End RAG Evaluation & Ablation

This notebook runs the **end-to-end RAG evaluation** and **ablation study** for MediQuery.

1. **Phase 6** — End-to-end RAG evaluation on 500 QA pairs (Baseline LLM, RAG, RAG + Query Rewrite)
2. **Phase 7** — Ablation study (isolating bi-encoder vs. reranker fine-tuning gains)

### Prerequisites

This notebook requires artifacts from **Step 4A** (`4A_Fine_Tuning.ipynb`):

| Artifact | Path |
|---|---|
| Fine-tuned bi-encoder | `{DATA_DIR}/bge-base-mediquery-finetuned/` |
| Fine-tuned FAISS index | `{DATA_DIR}/faiss_index/medicare_finetuned.index` |
| Fine-tuned reranker | `{DATA_DIR}/bge-reranker-mediquery-finetuned/` |
| Chunk corpus | `{DATA_DIR}/all_chunks.json` |
| Pre-trained FAISS index | `{DATA_DIR}/faiss_index/medicare.index` |
| Chunk metadata | `{DATA_DIR}/faiss_index/chunk_metadata.json` |
| Eval QA pairs | `{DATA_DIR}/datasets/rag_final_eval_500_qa_pairs_v2_claude.csv` |

## Install Dependencies

In [ ]:
!pip install -q sentence-transformers faiss-cpu pandas scikit-learn

## Mount Google Drive & Path Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os

# --- PATH CONFIGURATION (matches Step 4A notebook) ---
DATA_DIR    = '/content/drive/MyDrive/Embeddings'
CHUNKS_FILE = f'{DATA_DIR}/all_chunks.json'
OUTPUT_DIR  = f'{DATA_DIR}/faiss_index'
INDEX_FILE  = f'{OUTPUT_DIR}/medicare.index'
META_FILE   = f'{OUTPUT_DIR}/chunk_metadata.json'

# --- FINE-TUNED MODEL PATHS (produced by Step 4A) ---
FT_BIENCODER_DIR = f'{DATA_DIR}/bge-base-mediquery-finetuned'
FT_RERANKER_DIR  = f'{DATA_DIR}/bge-reranker-mediquery-finetuned'
FT_INDEX_FILE    = f'{OUTPUT_DIR}/medicare_finetuned.index'
FT_EMBED_FILE    = f'{OUTPUT_DIR}/embeddings_finetuned.npy'

# --- EVAL DATA ---
DATASETS_DIR = f'{DATA_DIR}'
EVAL_CSV     = f'{DATASETS_DIR}/rag_final_eval_500_qa_pairs_v2_claude.csv'

# --- Verify all required artifacts exist ---
required = {
    'Chunk corpus':            CHUNKS_FILE,
    'Chunk metadata':          META_FILE,
    'Pre-trained FAISS index': INDEX_FILE,
    'Fine-tuned bi-encoder':   FT_BIENCODER_DIR,
    'Fine-tuned FAISS index':  FT_INDEX_FILE,
    'Fine-tuned reranker':     FT_RERANKER_DIR,
    'Eval CSV':                EVAL_CSV,
}

print('Artifact check:')
all_ok = True
for name, path in required.items():
    exists = os.path.exists(path)
    status = 'OK' if exists else 'MISSING'
    if not exists:
        all_ok = False
    print(f'  {status:7s} | {name:28s} | {path}')

if not all_ok:
    print('\nERROR: Some artifacts are missing. Run Step 4A first.')
else:
    print('\nAll artifacts found.')

## Load All Models and Data

Load every artifact needed for evaluation:
- Chunk corpus + metadata
- Pre-trained bi-encoder + FAISS index (for ablation)
- Fine-tuned bi-encoder + FAISS index
- Pre-trained reranker (for ablation)
- Fine-tuned reranker
- 500-question eval set

In [ ]:
import json
import numpy as np
import pandas as pd
import faiss
import torch
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# --- Chunk corpus and metadata ---
with open(CHUNKS_FILE, 'r', encoding='utf-8') as f:
    all_chunks = json.load(f)
with open(META_FILE, 'r', encoding='utf-8') as f:
    chunk_metadata = json.load(f)
print(f'Chunks loaded: {len(all_chunks):,}')

# --- Pre-trained models (for ablation comparison) ---
pretrained_embed_model = SentenceTransformer('BAAI/bge-base-en-v1.5', device=device)
pretrained_index = faiss.read_index(INDEX_FILE)
pretrained_reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', device=device, max_length=512)
print(f'Pre-trained bi-encoder loaded: BAAI/bge-base-en-v1.5')
print(f'Pre-trained FAISS index: {pretrained_index.ntotal:,} vectors')
print(f'Pre-trained reranker loaded: BAAI/bge-reranker-v2-m3')

# --- Fine-tuned models (from Step 4A) ---
finetuned_embed_model = SentenceTransformer(FT_BIENCODER_DIR, device=device)
finetuned_index = faiss.read_index(FT_INDEX_FILE)
finetuned_reranker = CrossEncoder(FT_RERANKER_DIR, device=device, max_length=512)
print(f'Fine-tuned bi-encoder loaded from: {FT_BIENCODER_DIR}')
print(f'Fine-tuned FAISS index: {finetuned_index.ntotal:,} vectors')
print(f'Fine-tuned reranker loaded from: {FT_RERANKER_DIR}')

# --- Eval dataset ---
eval_df = pd.read_csv(EVAL_CSV)
print(f'\nEval set loaded: {len(eval_df)} questions')
print(eval_df['question_type'].value_counts())

## Helper Functions

In [ ]:
def retrieve_chunks(query, embed_model, index, all_chunks, chunk_metadata, top_k=20):
    """Embed a query and retrieve top-k chunks from a FAISS index."""
    query_vec = embed_model.encode(
        query, normalize_embeddings=True, convert_to_numpy=True
    ).astype('float32').reshape(1, -1)

    scores, indices = index.search(query_vec, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        if idx == -1:
            continue
        meta = chunk_metadata[idx]
        results.append({
            'faiss_score': float(scores[0][rank]),
            'text':        all_chunks[idx]['text'],
            'title':       meta['title'],
            'type':        meta['type'],
            'states':      meta.get('states', ['ALL']),
            'source_id':   meta['source_id'],
            'chunk_idx':   meta['chunk_idx'],
            'chunk_id':    f"{meta['source_id']}_{meta['chunk_idx']}"
        })
    return results


def filter_by_state(results, state=None):
    """Filter retrieved chunks to those covering a specific state."""
    if state is None:
        return results
    return [r for r in results if 'ALL' in r['states'] or state in r['states']]


def rerank_results(query, results, reranker_model, top_n=5, deduplicate=False):
    """Rerank retrieved chunks using a cross-encoder.

    Args:
        deduplicate: If True, keep only one chunk per source_id (for RAG generation).
                     If False, keep all chunks (for retrieval evaluation).
    """
    if not results:
        return []

    pairs = [(query, r['text']) for r in results]
    scores = reranker_model.predict(pairs)

    for i in range(len(results)):
        results[i]['rerank_score'] = float(scores[i])

    results.sort(key=lambda x: x['rerank_score'], reverse=True)

    if not deduplicate:
        return results[:top_n]

    seen = set()
    final = []
    for r in results:
        if r['source_id'] not in seen:
            final.append(r)
            seen.add(r['source_id'])
        if len(final) == top_n:
            break
    return final


def retrieve_and_rerank(query, embed_model, faiss_index, reranker_model,
                        all_chunks, chunk_metadata, state=None,
                        top_k_retrieve=20, top_n_rerank=5):
    """Full retrieval pipeline: embed -> FAISS -> state filter -> rerank."""
    retrieved = retrieve_chunks(query, embed_model, faiss_index,
                                all_chunks, chunk_metadata, top_k=top_k_retrieve)
    filtered = filter_by_state(retrieved, state)
    reranked = rerank_results(query, filtered, reranker_model, top_n=top_n_rerank)
    return reranked


def evaluate_retrieval(embed_model, faiss_index, reranker_model,
                       all_chunks, chunk_metadata, eval_df,
                       top_k_retrieve=20, top_n_rerank=5):
    """Evaluate retrieval: Recall@5/10/20, MRR, before and after reranking."""
    results_per_row = []

    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc='Evaluating'):
        question = row['question']
        gold_chunk_id = str(row['chunk_id'])

        retrieved = retrieve_chunks(
            question, embed_model, faiss_index,
            all_chunks, chunk_metadata, top_k=top_k_retrieve
        )
        retrieved_ids = [r['chunk_id'] for r in retrieved]

        # Rerank to top-5 (no dedup for fair eval comparison)
        reranked = rerank_results(question, retrieved, reranker_model, top_n=top_n_rerank, deduplicate=False)
        reranked_ids = [r['chunk_id'] for r in reranked]

        hit_at_5  = 1 if gold_chunk_id in retrieved_ids[:5]  else 0
        hit_at_10 = 1 if gold_chunk_id in retrieved_ids[:10] else 0
        hit_at_20 = 1 if gold_chunk_id in retrieved_ids[:20] else 0

        if gold_chunk_id in retrieved_ids:
            mrr = 1.0 / (retrieved_ids.index(gold_chunk_id) + 1)
        else:
            mrr = 0.0

        hit_at_5_reranked = 1 if gold_chunk_id in reranked_ids else 0
        if gold_chunk_id in reranked_ids:
            mrr_reranked = 1.0 / (reranked_ids.index(gold_chunk_id) + 1)
        else:
            mrr_reranked = 0.0

        results_per_row.append({
            'qa_id':              row.get('qa_id', ''),
            'question_type':      row.get('question_type', ''),
            'doc_type':           row.get('doc_type', ''),
            'coverage_status':    row.get('coverage_status', ''),
            'hit_at_5':           hit_at_5,
            'hit_at_10':          hit_at_10,
            'hit_at_20':          hit_at_20,
            'mrr':                mrr,
            'hit_at_5_reranked':  hit_at_5_reranked,
            'mrr_reranked':       mrr_reranked,
        })

    results_df = pd.DataFrame(results_per_row)
    summary = {
        'Recall@5 (FAISS)':    results_df['hit_at_5'].mean(),
        'Recall@10 (FAISS)':   results_df['hit_at_10'].mean(),
        'Recall@20 (FAISS)':   results_df['hit_at_20'].mean(),
        'MRR (FAISS)':         results_df['mrr'].mean(),
        'Recall@5 (reranked)': results_df['hit_at_5_reranked'].mean(),
        'MRR (reranked)':      results_df['mrr_reranked'].mean(),
    }
    return summary, results_df


print('Helper functions defined.')

---
# Phase 6: End-to-End RAG Evaluation on 500 QA Pairs

**Note:** This phase requires the LLM generation pipeline (Mistral-7B-Instruct). If the generation code from Step 4 of the main pipeline is not yet available, this section can be completed once it is ready. The retrieval evaluation in Phase 5 can stand alone.

### Step 6.1 — Define system configurations

| Config | Retrieval | Reranker | Generator |
|---|---|---|---|
| **Baseline LLM** | None | None | Mistral-7B-Instruct (no context) |
| **RAG (fine-tuned)** | Fine-tuned BGE → FAISS → top-20 → Fine-tuned reranker → top-5 | Fine-tuned | Mistral-7B-Instruct |
| **RAG + Query Rewrite** | Same as above, with query rewriting step | Fine-tuned | Mistral-7B-Instruct |

In [ ]:
# ─── PLACEHOLDER: LLM Generation Functions ────────────────────────────────────
# Replace these with your actual Mistral-7B-Instruct generation code from Step 4.
#
# def generate_baseline(question):
#     """Generate an answer using Mistral-7B with NO retrieved context."""
#     prompt = f"Answer this Medicare question:\n{question}\nAnswer:"
#     return call_mistral(prompt)
#
# def generate_with_context(question, chunks):
#     """Generate a grounded answer using retrieved chunks as context."""
#     context = "\n\n".join([c['text'] for c in chunks])
#     prompt = f"""Based ONLY on the following evidence, answer the question.
#     If the evidence is insufficient, say so.
#
#     Evidence:
#     {context}
#
#     Question: {question}
#     Answer:"""
#     return call_mistral(prompt)
#
# def rewrite_query(question):
#     """Use the LLM to rewrite the question for better retrieval."""
#     prompt = f"Rewrite this question as a search query:\n{question}\nSearch query:"
#     return call_mistral(prompt)

print("LLM generation functions: PLACEHOLDER")
print("Replace with your Mistral-7B-Instruct code before running Phase 6.")

### Step 6.2 — Run all 500 queries through each configuration

Uncomment and run after integrating the LLM generation code.

In [ ]:
# import re
#
# results_all = []
#
# for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="E2E Eval"):
#     question    = row['question']
#     gold_answer = row['answer']
#     gold_chunk  = str(row['chunk_id'])
#
#     # ─── Config 1: Baseline LLM (no retrieval) ────────────────────────────
#     baseline_answer = generate_baseline(question)
#
#     # ─── Config 2: RAG (fine-tuned bi-encoder + fine-tuned reranker) ──────
#     rag_chunks = retrieve_and_rerank(
#         question, finetuned_embed_model, finetuned_index, finetuned_reranker,
#         all_chunks, chunk_metadata
#     )
#     rag_answer = generate_with_context(question, rag_chunks)
#
#     # ─── Config 3: RAG + Query Rewrite ───────────────────────────────────
#     rewritten_q = rewrite_query(question)
#     rqr_chunks = retrieve_and_rerank(
#         rewritten_q, finetuned_embed_model, finetuned_index, finetuned_reranker,
#         all_chunks, chunk_metadata
#     )
#     rqr_answer = generate_with_context(question, rqr_chunks)
#
#     # ─── Collect results ──────────────────────────────────────────────────
#     results_all.append({
#         'qa_id':            row['qa_id'],
#         'question':         question,
#         'question_type':    row['question_type'],
#         'doc_type':         row['doc_type'],
#         'gold_answer':      gold_answer,
#         'gold_chunk_id':    gold_chunk,
#         'baseline_answer':  baseline_answer,
#         'rag_answer':       rag_answer,
#         'rag_chunk_ids':    [c['chunk_id'] for c in rag_chunks],
#         'rqr_answer':       rqr_answer,
#         'rqr_chunk_ids':    [c['chunk_id'] for c in rqr_chunks],
#     })
#
# results_e2e_df = pd.DataFrame(results_all)
# print(f"\nEnd-to-end evaluation complete: {len(results_e2e_df)} questions")
#
# # Save results
# results_e2e_df.to_csv(f'{DATA_DIR}/e2e_eval_results.csv', index=False)
# print(f"Results saved to: {DATA_DIR}/e2e_eval_results.csv")

### Step 6.3 — Evaluation metric functions

In [ ]:
import re

def extract_cited_ids(answer_text):
    """
    Extract NCD/LCD IDs cited in a generated answer.
    Looks for patterns like 'NCD 30.3.3', 'LCD L35125', 'NCD_ID: 30.3.3', etc.
    """
    patterns = [
        r'NCD\s*[:#]?\s*([\d.]+)',           # NCD 30.3.3 or NCD: 30.3.3
        r'LCD\s*[:#]?\s*L?(\d+)',            # LCD L35125 or LCD 35125
        r'NCD_ID:\s*([\d.]+)',                # NCD_ID: 30.3.3
        r'LCD_ID:\s*(\d+)',                   # LCD_ID: 35125
    ]
    cited = set()
    for pattern in patterns:
        matches = re.findall(pattern, answer_text, re.IGNORECASE)
        cited.update(matches)
    return cited


def check_citation_accuracy(generated_answer, gold_source_id):
    """Check if the generated answer cites the correct source document."""
    cited = extract_cited_ids(generated_answer)
    gold_id = str(gold_source_id)
    return 1 if gold_id in cited else 0


print("Evaluation metric functions defined.")
print(f"Test: extract_cited_ids('According to NCD 30.3.3 and LCD L35125...') = "
      f"{extract_cited_ids('According to NCD 30.3.3 and LCD L35125...')}")

### Step 6.4 — Compute and display results

Uncomment after running Step 6.2.

In [ ]:
# # ─── Compute metrics for each configuration ──────────────────────────────────
#
# def compute_generation_metrics(results_df, answer_col, chunk_ids_col=None):
#     """Compute citation accuracy and retrieval hit rate for a config."""
#     metrics = {}
#
#     # Citation accuracy
#     if answer_col in results_df.columns:
#         results_df[f'{answer_col}_citation'] = results_df.apply(
#             lambda r: check_citation_accuracy(str(r[answer_col]), r['gold_chunk_id']),
#             axis=1
#         )
#         metrics['citation_accuracy'] = results_df[f'{answer_col}_citation'].mean()
#
#     # Retrieval recall (if chunk IDs available)
#     if chunk_ids_col and chunk_ids_col in results_df.columns:
#         results_df[f'{chunk_ids_col}_hit'] = results_df.apply(
#             lambda r: 1 if r['gold_chunk_id'] in r[chunk_ids_col] else 0,
#             axis=1
#         )
#         metrics['recall_at_5'] = results_df[f'{chunk_ids_col}_hit'].mean()
#
#     return metrics
#
#
# baseline_metrics = compute_generation_metrics(results_e2e_df, 'baseline_answer')
# rag_metrics      = compute_generation_metrics(results_e2e_df, 'rag_answer', 'rag_chunk_ids')
# rqr_metrics      = compute_generation_metrics(results_e2e_df, 'rqr_answer', 'rqr_chunk_ids')
#
# print("\n" + "=" * 85)
# print("END-TO-END RAG EVALUATION RESULTS")
# print("=" * 85)
# print(f"{'System':25s} | {'Recall@5':>10s} | {'Citation Acc.':>14s}")
# print("-" * 85)
# print(f"{'Baseline LLM':25s} | {'—':>10s} | {baseline_metrics.get('citation_accuracy', 0):>13.4f}")
# print(f"{'RAG (fine-tuned)':25s} | {rag_metrics.get('recall_at_5', 0):>10.4f} | {rag_metrics.get('citation_accuracy', 0):>13.4f}")
# print(f"{'RAG + Query Rewrite':25s} | {rqr_metrics.get('recall_at_5', 0):>10.4f} | {rqr_metrics.get('citation_accuracy', 0):>13.4f}")
# print("=" * 85)

---
# Phase 7: Ablation — Pre-trained vs. Fine-tuned Component Comparison

Run retrieval evaluation with all 4 combinations of pre-trained/fine-tuned bi-encoder and reranker to isolate the contribution of each.

In [ ]:
ablation_configs = {
    'A: PT bienc + PT reranker': (pretrained_embed_model, pretrained_index, pretrained_reranker),
    'B: FT bienc + PT reranker': (finetuned_embed_model,  finetuned_index,  pretrained_reranker),
    'C: PT bienc + FT reranker': (pretrained_embed_model, pretrained_index, finetuned_reranker),
    'D: FT bienc + FT reranker': (finetuned_embed_model,  finetuned_index,  finetuned_reranker),
}

ablation_results = {}

for name, (emb_model, idx, reranker) in ablation_configs.items():
    print(f"\nRunning: {name}")
    summary, details = evaluate_retrieval(
        emb_model, idx, reranker,
        all_chunks, chunk_metadata, eval_df
    )
    ablation_results[name] = summary
    print(f"  Recall@5 (reranked): {summary['Recall@5 (reranked)']:.4f}")
    print(f"  MRR (reranked):      {summary['MRR (reranked)']:.4f}")

In [ ]:
print("\n" + "=" * 100)
print("ABLATION STUDY: Component-wise Fine-tuning Impact")
print("=" * 100)

metrics_to_show = ['Recall@5 (FAISS)', 'Recall@20 (FAISS)', 'MRR (FAISS)',
                   'Recall@5 (reranked)', 'MRR (reranked)']

config_names = list(ablation_results.keys())

# Header
header = f"{'Metric':28s}"
for name in config_names:
    header += f" | {name:>16s}"
print(header)
print("-" * 100)

# Rows
for metric in metrics_to_show:
    row = f"  {metric:26s}"
    for name in config_names:
        val = ablation_results[name].get(metric, 0)
        row += f" | {val:>15.4f}"
    print(row)

print("=" * 100)
print("\nPT = Pre-trained, FT = Fine-tuned")
print("Compare B vs A to see bi-encoder fine-tuning impact.")
print("Compare C vs A to see reranker fine-tuning impact.")
print("Compare D vs A to see combined fine-tuning impact.")

---
## Save All Fine-Tuned Artifacts Summary

| Artifact | Path |
|---|---|
| Fine-tuned bi-encoder | `{FT_BIENCODER_DIR}` |
| Fine-tuned FAISS index | `{FT_INDEX_FILE}` |
| Fine-tuned embeddings | `{FT_EMBED_FILE}` |
| Fine-tuned reranker | `{FT_RERANKER_DIR}` |
| Evaluation results | `{DATA_DIR}/e2e_eval_results.csv` |

In [ ]:
print("=" * 60)
print("SAVED ARTIFACTS")
print("=" * 60)

artifacts = {
    'Fine-tuned bi-encoder':  FT_BIENCODER_DIR,
    'Fine-tuned FAISS index': FT_INDEX_FILE,
    'Fine-tuned embeddings':  FT_EMBED_FILE,
    'Fine-tuned reranker':    FT_RERANKER_DIR,
}

for name, path in artifacts.items():
    exists = os.path.exists(path)
    if exists and os.path.isfile(path):
        size = os.path.getsize(path) / (1024 * 1024)
        print(f"  {name:30s}: {path}  ({size:.1f} MB)")
    elif exists:
        print(f"  {name:30s}: {path}  (directory)")
    else:
        print(f"  {name:30s}: NOT FOUND — {path}")

print("=" * 60)